In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox


DATABASE_NAME = "registrar.db"


# ============================================================
# DATABASE CLASS
# ============================================================

class RegistrarDatabase:
    def __init__(self, database_name=DATABASE_NAME):
        self.database_name = database_name
        self.create_tables()
        self.insert_sample_data_if_empty()

    def connect(self):
        """Create a connection and enable foreign-key constraints."""
        connection = sqlite3.connect(self.database_name)
        connection.execute("PRAGMA foreign_keys = ON")
        return connection

    def create_tables(self):
        """Create the three related tables if they do not already exist."""
        with self.connect() as connection:
            cursor = connection.cursor()

            cursor.execute("""
                CREATE TABLE IF NOT EXISTS students (
                    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    name TEXT NOT NULL,
                    major TEXT NOT NULL
                )
            """)

            cursor.execute("""
                CREATE TABLE IF NOT EXISTS enrollments (
                    enrollment_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    student_id INTEGER NOT NULL,
                    course_code TEXT NOT NULL,
                    semester TEXT NOT NULL,
                    FOREIGN KEY (student_id)
                        REFERENCES students(student_id)
                        ON DELETE CASCADE
                )
            """)

            cursor.execute("""
                CREATE TABLE IF NOT EXISTS grades (
                    grade_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    enrollment_id INTEGER NOT NULL,
                    assignment_name TEXT NOT NULL,
                    score REAL NOT NULL CHECK (score >= 0 AND score <= 100),
                    FOREIGN KEY (enrollment_id)
                        REFERENCES enrollments(enrollment_id)
                        ON DELETE CASCADE
                )
            """)

    def insert_sample_data_if_empty(self):
        """Insert demonstration records only when the students table is empty."""
        with self.connect() as connection:
            cursor = connection.cursor()
            cursor.execute("SELECT COUNT(*) FROM students")
            student_count = cursor.fetchone()[0]

            if student_count == 0:
                students = [
                    ("Aarav Sharma", "Computer Science"),
                    ("Diya Rao", "Mathematics"),
                    ("Rohan Kumar", "Data Science")
                ]
                cursor.executemany(
                    "INSERT INTO students (name, major) VALUES (?, ?)",
                    students
                )

                enrollments = [
                    (1, "CS-101", "Semester 1"),
                    (1, "DS-201", "Semester 1"),
                    (2, "MATH-202", "Semester 1"),
                    (3, "DS-201", "Semester 1")
                ]
                cursor.executemany("""
                    INSERT INTO enrollments
                    (student_id, course_code, semester)
                    VALUES (?, ?, ?)
                """, enrollments)

                grades = [
                    (1, "Assignment 1", 88),
                    (1, "Quiz 1", 92),
                    (2, "Assignment 1", 85),
                    (3, "Assignment 1", 91),
                    (4, "Assignment 1", 95)
                ]
                cursor.executemany("""
                    INSERT INTO grades
                    (enrollment_id, assignment_name, score)
                    VALUES (?, ?, ?)
                """, grades)

    def add_student_enrollment_grade(
        self, name, major, course_code, semester, assignment_name, score
    ):
        """Find an existing student by name or create a new student, then add records."""
        with self.connect() as connection:
            cursor = connection.cursor()

            cursor.execute("""
                SELECT student_id
                FROM students
                WHERE LOWER(name) = LOWER(?)
                ORDER BY student_id
                LIMIT 1
            """, (name,))
            result = cursor.fetchone()

            if result:
                student_id = result[0]
            else:
                cursor.execute("""
                    INSERT INTO students (name, major)
                    VALUES (?, ?)
                """, (name, major))
                student_id = cursor.lastrowid

            cursor.execute("""
                INSERT INTO enrollments
                (student_id, course_code, semester)
                VALUES (?, ?, ?)
            """, (student_id, course_code, semester))
            enrollment_id = cursor.lastrowid

            cursor.execute("""
                INSERT INTO grades
                (enrollment_id, assignment_name, score)
                VALUES (?, ?, ?)
            """, (enrollment_id, assignment_name, score))

            return student_id, enrollment_id

    def get_transcript(self):
        with self.connect() as connection:
            cursor = connection.cursor()
            cursor.execute("""
                SELECT
                    students.name,
                    students.major,
                    enrollments.course_code,
                    enrollments.semester,
                    grades.assignment_name,
                    grades.score
                FROM students
                INNER JOIN enrollments
                    ON students.student_id = enrollments.student_id
                INNER JOIN grades
                    ON enrollments.enrollment_id = grades.enrollment_id
                ORDER BY students.student_id, enrollments.enrollment_id,
                         grades.grade_id
            """)
            return cursor.fetchall()

    def get_all(self, table_name):
        """Fetch rows from one of the three known tables."""
        allowed_tables = {"students", "enrollments", "grades"}
        if table_name not in allowed_tables:
            raise ValueError("Invalid table name.")

        with self.connect() as connection:
            cursor = connection.cursor()
            cursor.execute(f"SELECT * FROM {table_name}")
            return cursor.fetchall()

    def get_student_options(self):
        with self.connect() as connection:
            cursor = connection.cursor()
            cursor.execute("""
                SELECT student_id, name
                FROM students
                ORDER BY name
            """)
            return cursor.fetchall()

    def get_enrollment_options(self):
        with self.connect() as connection:
            cursor = connection.cursor()
            cursor.execute("""
                SELECT
                    enrollments.enrollment_id,
                    students.name,
                    enrollments.course_code,
                    enrollments.semester
                FROM enrollments
                INNER JOIN students
                    ON enrollments.student_id = students.student_id
                ORDER BY students.name, enrollments.course_code
            """)
            return cursor.fetchall()

    def delete_student(self, student_id):
        with self.connect() as connection:
            cursor = connection.cursor()
            cursor.execute(
                "DELETE FROM students WHERE student_id = ?",
                (student_id,)
            )
            return cursor.rowcount

    def delete_enrollment(self, enrollment_id):
        with self.connect() as connection:
            cursor = connection.cursor()
            cursor.execute(
                "DELETE FROM enrollments WHERE enrollment_id = ?",
                (enrollment_id,)
            )
            return cursor.rowcount


# ============================================================
# TKINTER APPLICATION
# ============================================================

class RegistrarApplication:
    def __init__(self, root):
        self.root = root
        self.root.title("Registrar Operations - Academic Dashboard")
        self.root.geometry("1250x780")
        self.root.minsize(1000, 650)
        self.root.configure(bg="#edf2f7")

        self.database = RegistrarDatabase()

        self.create_styles()
        self.create_header()
        self.create_insert_panel()
        self.create_delete_panel()
        self.create_tabs()
        self.refresh_all_tables()

    def create_styles(self):
        style = ttk.Style()
        try:
            style.theme_use("clam")
        except tk.TclError:
            pass

        style.configure(
            "Treeview",
            rowheight=27,
            font=("Arial", 10)
        )
        style.configure(
            "Treeview.Heading",
            font=("Arial", 10, "bold")
        )
        style.configure(
            "TNotebook.Tab",
            font=("Arial", 10, "bold"),
            padding=(12, 7)
        )

    def create_header(self):
        header = tk.Frame(self.root, bg="#17365d", height=65)
        header.pack(fill="x")

        title = tk.Label(
            header,
            text="REGISTRAR OPERATIONS",
            font=("Arial", 21, "bold"),
            bg="#17365d",
            fg="white"
        )
        title.pack(pady=(9, 0))

        subtitle = tk.Label(
            header,
            text="Student Course Enrollment & Academic Performance Dashboard",
            font=("Arial", 10),
            bg="#17365d",
            fg="#dce6f1"
        )
        subtitle.pack(pady=(2, 8))

    def create_insert_panel(self):
        panel = tk.LabelFrame(
            self.root,
            text=" Registrar Insert Panel ",
            font=("Arial", 11, "bold"),
            bg="white",
            padx=10,
            pady=8
        )
        panel.pack(fill="x", padx=12, pady=(10, 5))

        self.name_var = tk.StringVar()
        self.major_var = tk.StringVar()
        self.course_var = tk.StringVar()
        self.semester_var = tk.StringVar(value="Semester 1")
        self.assignment_var = tk.StringVar(value="Assignment 1")
        self.score_var = tk.StringVar()

        fields = [
            ("Student Name", self.name_var, 0),
            ("Major (new student)", self.major_var, 1),
            ("Course Code", self.course_var, 2),
            ("Semester", self.semester_var, 3),
            ("Assignment Name", self.assignment_var, 4),
            ("Score (0-100)", self.score_var, 5)
        ]

        for label_text, variable, column in fields:
            tk.Label(
                panel,
                text=label_text,
                bg="white",
                font=("Arial", 9, "bold")
            ).grid(row=0, column=column, padx=5, pady=(2, 4), sticky="w")

            entry = tk.Entry(
                panel,
                textvariable=variable,
                font=("Arial", 10),
                width=19
            )
            entry.grid(row=1, column=column, padx=5, pady=3, sticky="ew")

        panel.grid_columnconfigure(0, weight=1)
        panel.grid_columnconfigure(1, weight=1)
        panel.grid_columnconfigure(2, weight=1)
        panel.grid_columnconfigure(3, weight=1)
        panel.grid_columnconfigure(4, weight=1)
        panel.grid_columnconfigure(5, weight=1)

        button_frame = tk.Frame(panel, bg="white")
        button_frame.grid(
            row=2, column=0, columnspan=6, pady=(8, 2), sticky="w"
        )

        tk.Button(
            button_frame,
            text="Add Record",
            command=self.add_record,
            bg="#198754",
            fg="white",
            font=("Arial", 10, "bold"),
            width=14,
            cursor="hand2"
        ).pack(side="left", padx=5)

        tk.Button(
            button_frame,
            text="Clear Fields",
            command=self.clear_inputs,
            bg="#6c757d",
            fg="white",
            font=("Arial", 10, "bold"),
            width=14,
            cursor="hand2"
        ).pack(side="left", padx=5)

    def create_delete_panel(self):
        panel = tk.LabelFrame(
            self.root,
            text=" Registrar Delete Panel ",
            font=("Arial", 11, "bold"),
            bg="white",
            padx=10,
            pady=8
        )
        panel.pack(fill="x", padx=12, pady=5)

        tk.Label(
            panel,
            text="Delete Type:",
            bg="white",
            font=("Arial", 10, "bold")
        ).pack(side="left", padx=(5, 8))

        self.delete_type_var = tk.StringVar()
        self.delete_type_combo = ttk.Combobox(
            panel,
            textvariable=self.delete_type_var,
            values=(
                "Delete an Entire Student Record",
                "Cancel a Specific Course Enrollment"
            ),
            state="readonly",
            width=38
        )
        self.delete_type_combo.pack(side="left", padx=5)
        self.delete_type_combo.bind(
            "<<ComboboxSelected>>",
            self.update_delete_targets
        )

        tk.Label(
            panel,
            text="Select Record:",
            bg="white",
            font=("Arial", 10, "bold")
        ).pack(side="left", padx=(15, 8))

        self.delete_target_var = tk.StringVar()
        self.delete_target_combo = ttk.Combobox(
            panel,
            textvariable=self.delete_target_var,
            state="readonly",
            width=48
        )
        self.delete_target_combo.pack(side="left", padx=5)

        tk.Button(
            panel,
            text="Delete Selected",
            command=self.delete_selected,
            bg="#dc3545",
            fg="white",
            font=("Arial", 10, "bold"),
            width=15,
            cursor="hand2"
        ).pack(side="left", padx=10)

        self.delete_target_map = {}

    def create_tabs(self):
        tab_frame = tk.Frame(self.root, bg="#edf2f7")
        tab_frame.pack(fill="both", expand=True, padx=12, pady=(5, 12))

        self.notebook = ttk.Notebook(tab_frame)
        self.notebook.pack(fill="both", expand=True)

        self.transcript_tab = tk.Frame(self.notebook, bg="white")
        self.students_tab = tk.Frame(self.notebook, bg="white")
        self.enrollments_tab = tk.Frame(self.notebook, bg="white")
        self.grades_tab = tk.Frame(self.notebook, bg="white")

        self.notebook.add(
            self.transcript_tab, text="Master Academic Transcript"
        )
        self.notebook.add(self.students_tab, text="Students")
        self.notebook.add(self.enrollments_tab, text="Enrollments")
        self.notebook.add(self.grades_tab, text="Grades")

        self.transcript_tree = self.create_treeview(
            self.transcript_tab,
            ("Student Name", "Major", "Course Code", "Semester",
             "Assignment Name", "Score")
        )
        self.students_tree = self.create_treeview(
            self.students_tab,
            ("Student ID", "Name", "Major")
        )
        self.enrollments_tree = self.create_treeview(
            self.enrollments_tab,
            ("Enrollment ID", "Student ID", "Course Code", "Semester")
        )
        self.grades_tree = self.create_treeview(
            self.grades_tab,
            ("Grade ID", "Enrollment ID", "Assignment Name", "Score")
        )

    def create_treeview(self, parent, columns):
        container = tk.Frame(parent, bg="white")
        container.pack(fill="both", expand=True, padx=8, pady=8)

        tree = ttk.Treeview(
            container,
            columns=columns,
            show="headings",
            selectmode="browse"
        )

        for column in columns:
            tree.heading(column, text=column)
            tree.column(column, width=150, anchor="center")

        vertical_scrollbar = ttk.Scrollbar(
            container, orient="vertical", command=tree.yview
        )
        horizontal_scrollbar = ttk.Scrollbar(
            container, orient="horizontal", command=tree.xview
        )

        tree.configure(
            yscrollcommand=vertical_scrollbar.set,
            xscrollcommand=horizontal_scrollbar.set
        )

        tree.grid(row=0, column=0, sticky="nsew")
        vertical_scrollbar.grid(row=0, column=1, sticky="ns")
        horizontal_scrollbar.grid(row=1, column=0, sticky="ew")

        container.grid_rowconfigure(0, weight=1)
        container.grid_columnconfigure(0, weight=1)

        return tree

    # --------------------------------------------------------
    # INSERT OPERATION
    # --------------------------------------------------------

    def add_record(self):
        name = self.name_var.get().strip()
        major = self.major_var.get().strip()
        course_code = self.course_var.get().strip()
        semester = self.semester_var.get().strip()
        assignment_name = self.assignment_var.get().strip()
        score_text = self.score_var.get().strip()

        # Reject blank fields.
        if not all([
            name, major, course_code, semester,
            assignment_name, score_text
        ]):
            messagebox.showerror(
                "Input Error",
                "Please fill in all fields before adding a record."
            )
            return

        # Validate that score is numeric.
        try:
            score = float(score_text)
        except ValueError:
            messagebox.showerror(
                "Invalid Score",
                "Assignment score must be a valid number."
            )
            return

        # Validate score range.
        if score < 0 or score > 100:
            messagebox.showerror(
                "Invalid Score",
                "Score must be between 0 and 100."
            )
            return

        try:
            student_id, enrollment_id = (
                self.database.add_student_enrollment_grade(
                    name, major, course_code, semester,
                    assignment_name, score
                )
            )

            messagebox.showinfo(
                "Success",
                "Record added successfully!\n"
                f"Student ID: {student_id}\n"
                f"Enrollment ID: {enrollment_id}"
            )

            self.clear_inputs()
            self.refresh_all_tables()

        except sqlite3.Error as error:
            messagebox.showerror(
                "Database Error",
                f"Unable to add the record.\n\n{error}"
            )

    def clear_inputs(self):
        self.name_var.set("")
        self.major_var.set("")
        self.course_var.set("")
        self.semester_var.set("Semester 1")
        self.assignment_var.set("Assignment 1")
        self.score_var.set("")

    # --------------------------------------------------------
    # DELETE OPERATION
    # --------------------------------------------------------

    def update_delete_targets(self, event=None):
        delete_type = self.delete_type_var.get()
        self.delete_target_map = {}

        if delete_type == "Delete an Entire Student Record":
            records = self.database.get_student_options()

            for student_id, name in records:
                display_text = f"ID {student_id} - {name}"
                self.delete_target_map[display_text] = student_id

        elif delete_type == "Cancel a Specific Course Enrollment":
            records = self.database.get_enrollment_options()

            for enrollment_id, name, course, semester in records:
                display_text = (
                    f"Enrollment {enrollment_id} - "
                    f"{name} - {course} - {semester}"
                )
                self.delete_target_map[display_text] = enrollment_id

        options = list(self.delete_target_map.keys())
        self.delete_target_combo["values"] = options
        self.delete_target_var.set(options[0] if options else "")

    def delete_selected(self):
        delete_type = self.delete_type_var.get()
        selected_record = self.delete_target_var.get()

        if not delete_type:
            messagebox.showwarning(
                "Selection Required",
                "Please select a delete category."
            )
            return

        if not selected_record:
            messagebox.showwarning(
                "Selection Required",
                "No record is available to delete."
            )
            return

        record_id = self.delete_target_map.get(selected_record)
        if record_id is None:
            messagebox.showerror(
                "Selection Error",
                "Please select a valid record."
            )
            return

        confirm = messagebox.askyesno(
            "Confirm Deletion",
            "Are you sure you want to delete this record?\n"
            "Related child records will also be deleted automatically."
        )
        if not confirm:
            return

        try:
            if delete_type == "Delete an Entire Student Record":
                deleted = self.database.delete_student(record_id)
            else:
                deleted = self.database.delete_enrollment(record_id)

            if deleted:
                messagebox.showinfo(
                    "Success",
                    "Selected record deleted successfully."
                )
            else:
                messagebox.showwarning(
                    "Not Found",
                    "The selected record no longer exists."
                )

            self.refresh_all_tables()

        except sqlite3.Error as error:
            messagebox.showerror(
                "Database Error",
                f"Unable to delete the record.\n\n{error}"
            )

    # --------------------------------------------------------
    # REFRESH ALL FOUR DISPLAY TABLES
    # --------------------------------------------------------

    def fill_tree(self, tree, rows):
        tree.delete(*tree.get_children())
        for row in rows:
            tree.insert("", tk.END, values=row)

    def refresh_all_tables(self):
        self.fill_tree(
            self.transcript_tree,
            self.database.get_transcript()
        )
        self.fill_tree(
            self.students_tree,
            self.database.get_all("students")
        )
        self.fill_tree(
            self.enrollments_tree,
            self.database.get_all("enrollments")
        )
        self.fill_tree(
            self.grades_tree,
            self.database.get_all("grades")
        )

        # Update the delete dropdowns after database changes.
        if self.delete_type_var.get():
            self.update_delete_targets()


# ============================================================
# MAIN PROGRAM
# ============================================================

if __name__ == "__main__":
    root = tk.Tk()
    app = RegistrarApplication(root)
    root.mainloop()